# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [113]:
# Write your code below.

%load_ext dotenv
%dotenv 

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [114]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [115]:
import os
from glob import glob

# Write your code below.
all_parquet_files = glob(os.path.join(os.getenv("PRICE_DATA"), "**/**/*.parquet"))
ddf = dd.read_parquet(all_parquet_files)


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [116]:
# Write your code below.
import pandas as pd
import dask.dataframe as dd

#define meta

meta_df = ddf.head(0).assign(
    Close_lag_1=0.0,
    Adj_Close_lag_1=0.0, 
    returns=0.0, 
    hi_lo_range=0.0)

# Perform groupby

ddf_feat = (
    ddf.groupby('ticker', group_keys=False)
       .apply(
           lambda x: x.sort_values('Date', ascending=True)
                        .assign(
                            Close_lag_1 = lambda df: df["Close"].shift(1),
                            Adj_Close_lag_1 = lambda df: df["Adj Close"].shift(1),
                            returns = lambda df: (df['Close'] / df['Close'].shift(1))-1,
                            hi_lo_range = lambda df: df['High'] - df['Low']
                        ),
            meta=meta_df
    )
)




+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [117]:
# Write your code below.
df_feat = ddf_feat.compute()

# add a 10-day moving array
df_feat['returns_ma_10'] = df_feat.groupby('ticker')['returns'].transform(lambda x: x.rolling(window=10).mean())

# View results
print(df_feat.head())


             Date  Open  High   Low  Close  Adj Close  Volume   source ticker  \
122498 2004-11-18  6.00  6.00  5.85   5.85       5.85  4500.0  NGD.csv    NGD   
122499 2004-11-19  5.61  5.62  5.61   5.62       5.62  1600.0  NGD.csv    NGD   
122500 2004-11-22  5.75  5.75  5.70   5.70       5.70  1500.0  NGD.csv    NGD   
122501 2004-11-23  5.65  5.65  5.60   5.65       5.65  2200.0  NGD.csv    NGD   
122502 2004-11-24  5.70  5.70  5.70   5.70       5.70     0.0  NGD.csv    NGD   

        Year  Close_lag_1  Adj_Close_lag_1   returns  hi_lo_range  \
122498  2004          NaN              NaN       NaN         0.15   
122499  2004         5.85             5.85 -0.039316         0.01   
122500  2004         5.62             5.62  0.014235         0.05   
122501  2004         5.70             5.70 -0.008772         0.05   
122502  2004         5.65             5.65  0.008850         0.00   

        returns_ma_10  
122498            NaN  
122499            NaN  
122500            NaN  
12

Please comment: 

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?
Based on my research, No. Dask support rolling windows like moving average, rolling max/min, etc. Dask rolling windows, however, require sorted partions which dask may not function well and give incorrect results.
(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.